# Figure 6: histidine-rich internal peptide termini in intestinal fluid

This notebook regenerates all six panels of Figure 6 from `source_data/Fig6/` with the plot code of the original producer, runs the producer's post-processing to the v28f composite and the v30 step (page flattened; the panel b key reads "Human-assigned" from the plot code), recomputes every test and interval the figure and the text report (CapScan 347 acquisitions, 15 participants; PREDICT 93 patients), and checks every panel pixel by pixel against the original render.

Run from the repository root with `python paper.py build Fig6`. Pixel checks and the post-processing need the Söhne fonts (`FASTALAKE_FONT_DIR`, with `soehne-buch.otf`, `soehne-buch.ttf` and `soehne-halbfett.ttf`).

In [ ]:
from pathlib import Path
import sys
UNIT = Path.cwd()                 # the notebook runs with its own directory as cwd
REPO = UNIT.parents[1]
sys.path[:0] = [str(REPO), str(UNIT)]
SD = REPO / "source_data" / UNIT.name
OUT = REPO / "build" / UNIT.name   # rendered panels; gitignored
OUT.mkdir(parents=True, exist_ok=True)
from style import fl_style as fl
import panels
fl.init()
# Figure 6 was drawn with the producer's style of September 2026 (0.6 pt lines); restore it exactly.
import os
FONT = os.environ.get("FASTALAKE_FONT_DIR") or os.environ.get("FL_STYLE_DIR")
HAVE_FONT = panels.producer_style(FONT)
print("Söhne registered:", HAVE_FONT)

## Data

All tables come from `source_data/Fig6/` (data dictionary in its `README.md`). The figure is drawn on one 180 x 166 mm canvas, panel by panel, with one random generator (seed 6) shared by the point jitter of panels a and b and the bootstrap of panel c, exactly as in the producer.

In [ ]:
d = panels.load(SD)
fig, ax_mm, st = panels.new_figure()
print({k: len(v) for k, v in d.items()})

## Panel a

Fraction of 9-50-residue peptides with a non-K/R C terminus in regions 1-4 and stool for three routes; dashed line, conventional tryptic search. Supports v28d line 165: "Peptides ending in neither lysine nor arginine (non-K/R termini) accounted for 5.0–5.5% of sequences across the four intestinal sampling regions, compared with 3.1% in stool, with higher intestinal-fluid fractions in all fourteen paired participants" (v28f: 5.1–5.6%, fifteen; tryptic 0.34%). n = 15 participants per region (unit participant). Test: paired sign-flip test of capsule mean minus stool, Benjamini-Hochberg within the 4,536-test family.

In [ ]:
panels.panel_a(fig, ax_mm, st, d["pp"], d["fam"])
panels.panel_crop(fig, "a", OUT)

## Panel b

Histidine share among non-K/R C termini for human- and microbial-assigned peptides, non-specific search. Key: "Human-assigned" (v30 term), written by the plot code. Supports v28d line 173 (v30 paragraph 89): "in all fourteen participants, with mean paired differences of 15.2 and 18.0 percentage points" (v28f: fifteen; 15.2 and 17.6). n = 15 participants (unit participant). Test as panel a.

In [ ]:
panels.panel_b(fig, ax_mm, st, d["pp"], d["fam"])
panels.panel_crop(fig, "b", OUT)

## Panel c

Residue composition of non-K/R C termini in regions 1-2, regions 3-4 and stool (non-specific search, I/L kept distinct). Supports v28d line 173: "histidine accounted for 40.3% and 40.8% of non-K/R C termini in regions 1–2 and regions 3–4, respectively, compared with 24.4% in stool" (v28f: 39.9% and 40.3%). n = 15 participants per group (unit participant). Interval: 95% percentile bootstrap, 2,000 participant resamples (v28d line 869).

In [ ]:
panels.panel_c(fig, ax_mm, st, d["pp"], d["fam"])
panels.panel_crop(fig, "c", OUT)

## Panel d

Histidine at uniquely localized internal C-terminal boundaries, observed minus source-protein expectation, for CapScan regions 1-4 and stool and PREDICT stool. Supports v28d line 175: "approximately 40% of observed non-K/R C-terminal sites in intestinal fluid were histidines, compared with approximately 2.4% of eligible positions in the same proteins". n = 15 participants (CapScan), 93 patients (PREDICT). Symbols, means; bars, 95% bootstrap intervals (20,000 whole-participant or whole-patient resamples).

In [ ]:
panels.panel_d(fig, ax_mm, st, d["S"], d["R"])
panels.panel_crop(fig, "d", OUT)

## Panel e

As panel d for internal N-terminal boundaries. Supports v28d line 175: "Enrichment was greater at C-terminal than N-terminal boundaries and occurred in every intestinal sampling region". n and interval as panel d.

In [ ]:
panels.panel_e(fig, ax_mm, st, d["S"], d["R"])
panels.panel_crop(fig, "e", OUT)

## Panel f

Complementary-product rate at histidine boundaries minus matched non-histidine boundaries. Supports v28d line 177: "In intestinal fluid, histidine boundaries were more often accompanied by such complementary products than matched non-histidine boundaries across de novo, semi-tryptic and non-specific routes". n and interval as panel d; test, sign-flip with Benjamini-Hochberg within the 90-test CapScan family. This cell also draws the shared route legend.

In [ ]:
panels.panel_f(fig, ax_mm, st, d["S"], d["R"])
panels.panel_crop(fig, "f", OUT)

## Composite

The producer's chain after the builder: panel letters set in Söhne Halbfett, blank bands trimmed, export at 183 mm, region names replaced by Region 1-4 as PDF text overlays, final PyMuPDF re-save. `ASSEMBLY.md` lists each step.

v30 step: the page is written as one 600 dpi raster (`panels.relabel_and_flatten` with no relabel). The panel b key already reads "Human-assigned" from `panels.panel_b`; the v29 producer had instead covered and retyped the key "Host: ..." on the composite. The builder, panel b and v28f comparisons below therefore differ from the v28f references in the panel b key only.

In [ ]:
builder_pdf, builder_png = panels.save_builder(fig, OUT)
if HAVE_FONT:
    post = panels.postprocess(builder_pdf, OUT / "post", FONT)
    rel = panels.relabel_regions(post["pdf"], OUT, FONT)
    final_pdf, final_png = panels.final_resave(rel["pdf"], rel["png"], OUT)
    print("blank bands removed (72 dpi rows):", post["blank_bands_removed"], "page (pt):", post["page_pt"], "min font (pt):", post["min_font_pt"])
    print("region label overlays:", len(rel["edits"]))
    v30 = panels.relabel_and_flatten(final_pdf, OUT / "v30", FONT)
    print("v30 flatten raster (px):", v30["receipt"]["flat_pdf_raster_px"])
    # v33: route key raised 11 pt, 6 pt cut at the page bottom, 21 white rows cropped (producer Fig6_v31)
    panels.raise_route_key(fig)
    V33 = OUT / "v33"; b33_pdf, b33_png = panels.save_builder(fig, V33)
    post33 = panels.postprocess(b33_pdf, V33 / "post", FONT, crop_bottom_pt=panels.V33_CROP_BOTTOM_PT)
    rel33 = panels.relabel_regions(post33["pdf"], V33, FONT)
    f33_pdf, f33_png = panels.final_resave(rel33["pdf"], rel33["png"], V33)
    flat33 = panels.relabel_and_flatten(f33_pdf, V33 / "flat", FONT)
    v33 = panels.crop_bottom(flat33["pdf"], flat33["png"], V33 / "Figure6_v33.pdf", V33 / "Figure6_v33.png")
    print("v33 page (pt):", v33["page_pt"], "blank bands removed:", post33["blank_bands_removed"])
else:
    print("post-processing SKIPPED: fonts absent (needs soehne-*.ttf in FASTALAKE_FONT_DIR)")

## Statistics

Recomputes, from participant- and patient-level tables, the complete test families behind panels a-f with the moved upstream code, and compares them with the family tables the figure reads. Agreement is required to 1e-12 for P, q, means and interval bounds.

In [ ]:
import numpy as np, pandas as pd, platform
tf = panels.stat_terminal_families(d["pp"])
ref = d["fam"][d["fam"].family.isin(tf.family.unique())]
k24 = ["route", "min_length", "unit", "origin", "terminus", "metric", "contrast", "family"]
c24 = panels.compare_families(tf, ref, k24, ["p", "q", "mean_difference", "CI95_low", "CI95_high", "n_participants", "n_positive", "n_negative"])
print("A24 families", tf.groupby("family").size().to_dict()); print(c24)
m = tf.merge(ref, on=k24, suffixes=("", "_ref"))
diff = m[(m.n_positive != m.n_positive_ref) | (m.n_negative != m.n_negative_ref)]
print("rows whose sign counts differ:", len(diff))
print(diff[k24 + ["n_positive", "n_positive_ref", "n_negative", "n_negative_ref", "mean_difference", "p", "p_ref"]].to_string())
sf = panels.stat_histidine_source_family(d["pe"])
k28 = ["route", "side", "origin", "residue", "contrast"]
c28 = panels.compare_families(sf, d["a28"][d["a28"].family.eq("source_P1_all_residues")], k28,
                              ["p", "q", "mean", "CI95_low", "CI95_high", "n", "n_positive", "observed_fraction_mean", "expected_fraction_mean", "enrichment_CI95_low", "enrichment_CI95_high"])
print("A28 source family", len(sf)); print(c28)
rf = panels.stat_histidine_reciprocal_family(d["pr"])
cr = panels.compare_families(rf, d["a28"][d["a28"].family.eq("all_reciprocal_controls")], ["route", "origin", "contrast"], ["p", "q", "mean", "CI95_low", "CI95_high", "n"])
print("A28 reciprocal family", len(rf)); print(cr)
pt = panels.stat_predict_patient(d["predict_patient"])
cp = panels.compare_families(pt, pd.read_csv(SD / panels.F_PREDICT_SOURCE, sep="\t"), ["route", "side", "origin", "residue"], ["p", "mean", "CI95_low", "CI95_high", "n", "n_positive"])
print("PREDICT patient rows", len(pt)); print(cp)
pq = panels.predict_family_q(d["predict_fam"])
print("PREDICT families", pq.groupby("family").size().to_dict(), "max |q recomputed - q shipped|:", float((pq.q_recomputed - pq.q).abs().max()))
assert float((pq.q_recomputed - pq.q).abs().max()) <= 1e-12
k = ["endpoint", "route", "side", "origin", "residue", "contrast"]
prim = pd.read_csv(SD / panels.F_PREDICT_SOURCE, sep="\t").merge(d["predict_fam"], on=k, suffixes=("", "_fam"))
assert len(prim) == 12 and (prim.q == prim.q_fam).all() and (prim.p == prim.p_fam).all(), "PREDICT display rows are not rows of the family table"
for name, c in [("A24", c24), ("A28 source", c28), ("A28 reciprocal", cr), ("PREDICT", cp)]:
    assert c["rows_joined"] == c["rows_new"] == c["rows_ref"], name
    for col in ("p", "q", "mean", "mean_difference", "CI95_low", "CI95_high"):
        if col in c["max_abs_diff"]:
            assert c["max_abs_diff"][col][0] <= 1e-12 and c["max_abs_diff"][col][1] == 0, (name, col, c["max_abs_diff"][col])
print("all recomputed P, q, means and intervals agree with the shipped tables to 1e-12")
SOFTWARE = f"CPython {platform.python_version()}, numpy {np.__version__}, pandas {pd.__version__}"
print(SOFTWARE)

## Pixel checks and numbers

Each panel crop and the composite (builder PDF and PNG, v28f PDF and PNG) is compared with the original producer's render in `expected/`. Then every number in `NUMBERS.tsv` is printed in file order and compared with the `recomputed_value` column.

In [ ]:
import csv
from tools.check_pixels import compare
if HAVE_FONT:
    for x in "abcdef":
        r = compare(OUT / f"panel_{x}.png", UNIT / "expected" / f"panel_{x}.png", UNIT / "checks" / f"panel_{x}", dpi=300)
        print(f"panel {x}:", r["verdict"], r.get("differing_pixels"))
    for new, exp, name in [(builder_pdf, "builder_Figure6.pdf", "builder_pdf"), (builder_png, "builder_Figure6.png", "builder_png"),
                           (final_pdf, "Fig6_v28f.pdf", "Fig6_v28f_pdf"), (final_png, "Fig6_v28f.png", "Fig6_v28f_png"),
                           (v30["pdf"], "Fig6_v30.pdf", "Fig6_v30_pdf"), (v30["png"], "Fig6_v30.png", "Fig6_v30_png"),
                           (v33["pdf"], "Fig6_v33.pdf", "Fig6_v33_pdf"), (v33["png"], "Fig6_v33.png", "Fig6_v33_png")]:
        r = compare(new, UNIT / "expected" / exp, UNIT / "checks" / name, dpi=300)
        print(f"{name}:", r["verdict"], r.get("differing_pixels"))
else:
    print("pixel checks SKIPPED: fonts absent")
numbers = panels.collect_numbers(st["nums"], d, tf, sf, rf, pt)
with open(UNIT / "NUMBERS.tsv", newline="", encoding="utf-8") as fh:
    table = list(csv.DictReader(fh, delimiter="\t"))
assert len(table) == len(numbers), (len(table), len(numbers))
bad = 0
for (key, value), row in zip(numbers, table):
    shown = "" if value is None else (value if isinstance(value, str) else repr(value))
    ok = shown == row["recomputed_value"]
    bad += not ok
    print(f"{row['v28d_line']:>4} {row['printed_value']:>10} -> {shown:<24} {row['status']:<22} {'' if ok else 'DIFFERS FROM NUMBERS.tsv'}")
assert bad == 0, f"{bad} numbers differ from NUMBERS.tsv"
stats = panels.stats_table(st["nums"], d, tf, sf, rf, pt, SOFTWARE)
committed = pd.read_csv(UNIT / "STATS.tsv", sep="\t", dtype=str, keep_default_na=False)
assert committed.drop(columns="software_version").equals(stats.astype(str).drop(columns="software_version")), "STATS.tsv differs from recomputation"
print(len(stats), "STATS.tsv rows reproduced")